# Step 2 — A transferable bifurcation measurement, and what Var[H] really measures

**Result from Step 1.** The fixed-point bifurcation diagram (pitchfork at τ=1 for w=0.5; saddle-node at τ≈0.45 for w=0.8) is the ground truth, but it was computed from the *analytic* density, which a trained model doesn't give us. Also, Var[H(c|m)] peaked at τ≈0.71 and τ≈1.0 respectively, not at the bifurcations, and was blind to the spinodal.

**This notebook does two things.**

1. **Conditional-drift estimator.** Identity: `E[ v·score(x_t) | m = v·x_t ] = d/dm log p_tau(m)`. So if we forward-noise data, run the *denoiser* `D(x_t)`, project `(D(x_t) - x_t)/sigma^2` onto the class direction, and **bin by m**, the zero-crossings of the binned average reproduce the fixed-point diagram. This uses only samples + a denoiser, so it transfers directly to trained models. We validate it here in full `d` dimensions against the analytic diagram.

2. **What is Var[H]?** Hypothesis from Step 1: the Var[H] peak occurs at a roughly *universal* value of E[H] (~0.27 nats), i.e. it is a "half-commitment clock", not a transition detector. We test this across w = 0.5 … 0.95.

CPU only, ~3 minutes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1)
GRID = np.linspace(-4.0, 4.0, 8001)
plt.rcParams["figure.dpi"] = 110

def mixture_pdf(m, tau, w):
    g = lambda mu: np.exp(-(m - mu)**2/(2*tau))/np.sqrt(2*np.pi*tau)
    return w*g(1.0) + (1-w)*g(-1.0)

def stationary_points(tau, w, grid=GRID, tol=1e-4):
    """Analytic ground truth, with a tolerance to remove flat-region duplicates seen in Step 1."""
    dlogp = np.gradient(np.log(mixture_pdf(grid, tau, w)), grid)
    idx = np.where(np.sign(dlogp[:-1]) != np.sign(dlogp[1:]))[0]
    pts = []
    for i in idx:
        if max(abs(dlogp[i]), abs(dlogp[i+1])) < tol:
            continue
        root = grid[i] - dlogp[i]*(grid[i+1]-grid[i])/(dlogp[i+1]-dlogp[i])
        pts.append((float(root), 'max' if dlogp[i] > 0 else 'min'))
    return pts

def posterior_entropy(m, tau, w):
    a = np.log(w) - (m-1)**2/(2*tau); b = np.log(1-w) - (m+1)**2/(2*tau)
    p = np.clip(1/(1+np.exp(b-a)), 1e-12, 1-1e-12)
    return -(p*np.log(p) + (1-p)*np.log(1-p))

## 1. Conditional-drift estimator (full d, denoiser-based)

The analytic denoiser for this mixture is `D(x) = tanh(s·x/sigma^2 + h) * s`, `h = ½ log(w/(1-w))`. In Phase 2 this line is replaced by a network call; everything else stays the same.

In [ ]:
def denoiser(xt, s, sigma, w):
    h = 0.5*np.log(w/(1-w))
    return np.tanh(xt @ s / sigma**2 + h)[:, None] * s[None, :]

def conditional_drift(tau, w, d=256, n=200_000, nbins=60, m_range=(-2.5, 2.5)):
    """Returns bin centers, binned <v.score | m>, counts. Only samples + denoiser are used."""
    s = np.sign(rng.standard_normal(d))
    c = rng.random(n) < w
    x0 = np.where(c[:, None], s, -s)
    sigma = np.sqrt(tau*d)
    xt = x0 + sigma*rng.standard_normal((n, d))
    m = xt @ s / d
    # (D - x)/sigma^2 is the score in x; its projection onto s is d/dm log p(m) (marginal-score identity)
    score_m = ((denoiser(xt, s, sigma, w) - xt) @ s) / sigma**2
    edges = np.linspace(*m_range, nbins+1)
    idx = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(idx, minlength=nbins)
    mean = np.bincount(idx, weights=score_m, minlength=nbins) / np.maximum(cnt, 1)
    centers = 0.5*(edges[:-1] + edges[1:])
    return centers, mean, cnt

def zero_crossings(centers, mean, cnt, min_count=200):
    ok = cnt >= min_count
    c, f = centers[ok], mean[ok]
    out = []
    for i in range(len(c)-1):
        if np.sign(f[i]) != np.sign(f[i+1]) and f[i] != 0:
            root = c[i] - f[i]*(c[i+1]-c[i])/(f[i+1]-f[i])
            out.append((float(root), 'max' if f[i] > 0 else 'min'))
    return out

# Validate at a few tau against the analytic diagram
for w in [0.5, 0.8]:
    print(f"\n=== w={w}: MC conditional-drift zero crossings vs analytic stationary points ===")
    for tau in [0.2, 0.4, 0.5, 0.7, 0.9, 1.2, 2.0]:
        cen, mean, cnt = conditional_drift(tau, w)
        mc = [(round(p,3),k) for p,k in zero_crossings(cen, mean, cnt)]
        an = [(round(p,3),k) for p,k in stationary_points(tau, w)]
        print(f"tau={tau:4.2f}  MC: {mc}\n          analytic: {an}")

### MC bifurcation diagrams (compare with Step 1's analytic ones)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
TAUS = np.geomspace(0.1, 3.0, 40)
for ax, w in zip(axes, [0.5, 0.8]):
    for tau in TAUS:
        cen, mean, cnt = conditional_drift(tau, w, n=100_000)
        for p, k in zero_crossings(cen, mean, cnt):
            ax.plot(tau, p, 'o', ms=3, color='C0' if k=='max' else 'C3')
        for p, k in stationary_points(tau, w):
            ax.plot(tau, p, '_', ms=8, color='k', alpha=0.4)
    ax.set_xscale('log'); ax.set_xlabel('tau'); ax.set_ylabel('fixed point m*')
    ax.set_title(f'w={w}: MC (colored) vs analytic (grey)')
    ax.axvline(1.0 if w==0.5 else 0.4502, ls='--', color='k', lw=0.8)
plt.tight_layout(); plt.show()

## 2. Is the Var[H] peak a transition detector or a 'half-commitment clock'?

For each w, locate the Var[H] peak in tau and record E[H] there, plus the true bifurcation tau.

In [ ]:
def bifurcation_tau(w):
    if abs(w-0.5) < 1e-9:
        return 1.0
    lo, hi = 0.02, 1.0
    for _ in range(40):
        mid = 0.5*(lo+hi)
        if len(stationary_points(mid, w)) == 3: lo = mid
        else: hi = mid
    return lo

taus = np.geomspace(0.05, 5.0, 200)
print(f"{'w':>5} {'tau_bif':>8} {'tau_peak':>9} {'E[H]@peak':>10} {'Var[H]@peak':>12}  ratio tau_peak/tau_bif")
rows = []
for w in [0.5, 0.55, 0.6, 0.7, 0.8, 0.9, 0.95]:
    best = None
    for tau in taus:
        m = np.where(rng.random(200_000) < w, 1.0, -1.0) + np.sqrt(tau)*rng.standard_normal(200_000)
        H = posterior_entropy(m, tau, w)
        v = H.var()
        if best is None or v > best[1]: best = (tau, v, H.mean())
    tb = bifurcation_tau(w)
    rows.append((w, tb, *best))
    print(f"{w:5.2f} {tb:8.3f} {best[0]:9.3f} {best[2]:10.4f} {best[1]:12.5f}  {best[0]/tb:6.2f}")

fig, ax = plt.subplots(figsize=(5.5, 3.8))
ax.plot([r[0] for r in rows], [r[1] for r in rows], 'ko-', label='bifurcation tau (ground truth)')
ax.plot([r[0] for r in rows], [r[2] for r in rows], 'C1s-', label='Var[H] peak tau')
ax.set_xlabel('class weight w'); ax.set_ylabel('tau'); ax.set_yscale('log'); ax.legend()
plt.tight_layout(); plt.show()

## What to report back

1. The MC-vs-analytic comparison table and the two MC bifurcation plots. We need the estimator to recover the pitchfork at τ=1 and the fold at τ≈0.45 / m≈−0.74 from samples alone. Note where bins become too sparse (the `min_count` filter) — that tells us how many samples Phase 2 will need.
2. The Var[H] table: does `E[H]@peak` stay near ~0.27 across w while `tau_peak` and `tau_bif` diverge from each other? If yes, Var[H] is a commitment clock, and the paper's first claim is that information-theoretic detectors and dynamical bifurcations are different objects.